# 📘 CIFAR-10 Image Classification — ANN vs CNN

## A Hands-On Comparison of Two Network Families on CIFAR-10

This notebook walks through building an image classifier two different ways —
once with a plain **Artificial Neural Network (ANN)** and once with a
**Convolutional Neural Network (CNN)** — and then digs into *why* one
outperforms the other, plus how different training choices change the outcome.

🎯 **Goal:** follow the markdown explanations alongside the runnable code to
build intuition for the full deep-learning workflow, end to end.

> 🟢 Every experiment below — both architectures and all five extension tasks —
> is implemented as code you can actually run, not left as a "try it yourself"
> placeholder. A **T4 GPU runtime** (Runtime → Change runtime type) is
> recommended to keep training time reasonable.


# 🧠 Problem Statement

Train an image classifier on **CIFAR-10** using two approaches:

1. **Artificial Neural Network (ANN)**
2. **Convolutional Neural Network (CNN)**

and then compare them on:
- Test accuracy
- Loss curves
- Generalization (train vs. validation behavior)
- The impact of training strategy (dropout, batch norm, augmentation)

---
### 📦 CIFAR-10 Classes
Airplane, Automobile, Bird, Cat, Deer, Dog, Frog, Horse, Ship, Truck


In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

print("TensorFlow version :", tf.__version__)
print("GPU available      :", tf.config.list_physical_devices("GPU"))

# 📥 Load Dataset

**CIFAR-10** ships with `tf.keras.datasets` already split into train/test:
- 50,000 training images
- 10,000 test images
- each image is 32×32 pixels with 3 color channels


In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

print("Training set:", x_train.shape, "->", y_train.shape)
print("Test set    :", x_test.shape, "->", y_test.shape)

## 🖼️ A Quick Look at the Data

In [ ]:
plt.figure(figsize=(10, 5))
for idx in range(10):
    plt.subplot(2, 5, idx + 1)
    plt.imshow(x_train[idx])
    plt.title(class_names[y_train[idx][0]])
    plt.axis("off")
plt.tight_layout()
plt.show()

# 🧹 Preprocessing

Pixel intensities are rescaled from the **0–255** range down to **0–1**,
which keeps gradients well-behaved during training. The ANN additionally
needs every image **flattened** into a single 3,072-length vector
(32 × 32 × 3), since a dense layer has no concept of a 2D image — the CNN
keeps the original 3D shape so it can exploit spatial structure.


In [ ]:
x_train_norm = x_train / 255.0
x_test_norm = x_test / 255.0

x_train_flat = x_train_norm.reshape(x_train_norm.shape[0], -1)
x_test_flat = x_test_norm.reshape(x_test_norm.shape[0], -1)

print("Flattened shape (ANN input):", x_train_flat.shape)
print("Image shape (CNN input)    :", x_train_norm.shape)

# 🔹 Part 1: ANN Model (Baseline)

An ANN sees an image as nothing more than a long list of numbers — it has no
way to tell that two pixels are physically next to each other. Watching where
this falls short is exactly what motivates the CNN in Part 2.


In [ ]:
ann_model = models.Sequential([
    layers.Dense(512, activation="relu", input_shape=(3072,)),
    layers.Dropout(0.3),
    layers.Dense(256, activation="relu"),
    layers.Dense(10, activation="softmax"),
])

ann_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

ann_model.summary()

ann_history = ann_model.fit(
    x_train_flat, y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
)

In [ ]:
ann_test_loss, ann_test_acc = ann_model.evaluate(x_test_flat, y_test)
print(f"ANN test accuracy: {ann_test_acc:.4f}")

# 🔹 Part 2: CNN Model (Baseline)

A CNN keeps the 2D layout of the image intact and learns filters that slide
across it, picking up on local patterns:
- Convolutional layers extract features
- Pooling layers downsample while keeping the strongest signals
- Stacking these blocks builds up from simple edges to complex shapes

This spatial awareness is what gives CNNs their edge on image tasks.


In [ ]:
cnn_model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation="relu", input_shape=(32, 32, 3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(10, activation="softmax"),
])

cnn_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

cnn_model.summary()

cnn_history = cnn_model.fit(
    x_train_norm, y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
)

In [ ]:
cnn_test_loss, cnn_test_acc = cnn_model.evaluate(x_test_norm, y_test)
print(f"CNN test accuracy: {cnn_test_acc:.4f}")

## 📈 ANN vs CNN — Validation Accuracy Over 10 Epochs

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(ann_history.history["val_accuracy"], marker="o", label="ANN val accuracy")
plt.plot(cnn_history.history["val_accuracy"], marker="s", label="CNN val accuracy")
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("ANN vs CNN — Validation Accuracy over 10 Epochs")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# 🚀 Training Strategy Upgrade: Data Augmentation

Instead of changing the architecture, this strategy changes *what the model
sees during training* — random flips, rotations, and zooms are applied to
each batch on the fly. Because the network rarely sees the exact same image
twice, augmentation acts as a regularizer and tends to reduce overfitting.


In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

aug_cnn_model = models.Sequential([
    data_augmentation,
    layers.Conv2D(32, 3, activation="relu", input_shape=(32, 32, 3)),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(10, activation="softmax"),
])

aug_cnn_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

aug_cnn_model.summary()

# 📊 Baseline Comparison Table (ANN vs CNN)

In [ ]:
comparison = pd.DataFrame({
    "Model": ["ANN", "CNN"],
    "Test Accuracy": [ann_test_acc, cnn_test_acc],
    "Test Loss": [ann_test_loss, cnn_test_loss],
})
comparison

# 🎓 Extension Tasks — Fully Implemented

Each task below is real, executable code rather than an exercise left for later.

| Task | Idea | Section |
|---|---|---|
| 1. Deepen the ANN | Add more Dense layers to the baseline ANN | Task 1 |
| 2. Widen the CNN filters | Push 32→64→128 out to a 4th block | Task 2 |
| 3. Train for longer | Bump the CNN's epoch budget from 10 to 20 | Task 3 |
| 4. Add EarlyStopping | Let validation loss decide when to stop | Task 4 |
| 5. Actually train the augmented model | Run `aug_cnn_model` and compare it | Task 5 |


## ✅ Task 1 — Deepen the ANN

The baseline ANN (Dense(512) → Dense(256) → Dense(10)) gets expanded into a
4-hidden-layer network with progressively narrower widths and dropout after
each block. The question we're testing: does stacking more Dense layers help
a model that still has zero awareness of pixel neighborhoods?


In [ ]:
ann_deep_model = models.Sequential([
    layers.Dense(1024, activation="relu", input_shape=(3072,)),
    layers.Dropout(0.3),
    layers.Dense(512, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(128, activation="relu"),
    layers.Dense(10, activation="softmax"),
])

ann_deep_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

ann_deep_model.summary()

ann_deep_history = ann_deep_model.fit(
    x_train_flat, y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
)

ann_deep_test_loss, ann_deep_test_acc = ann_deep_model.evaluate(x_test_flat, y_test)
print(f"Deep ANN test accuracy: {ann_deep_test_acc:.4f}")

**Takeaway to expect:** extra Dense layers typically buy only a *small*
accuracy improvement, and can even backfire (overfitting, vanishing
gradients) — because the network is still reasoning over a flattened
3,072-value vector with no sense of which pixels are adjacent. This is the
core reason CNNs were invented for vision tasks.


## ✅ Task 2 — Widen the CNN (4th Convolutional Block)

The baseline already scales 32→64→128 filters across its three blocks. Here
we add a 4th block to reach 32→64→128→256, to check whether more filter
capacity keeps paying off or whether a 32×32 input simply runs out of room
to keep being downsampled productively.


In [ ]:
cnn_scaled_model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation="relu", padding="same", input_shape=(32, 32, 3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(256, (3, 3), activation="relu", padding="same"),
    layers.BatchNormalization(),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(10, activation="softmax"),
])

cnn_scaled_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

cnn_scaled_model.summary()

cnn_scaled_history = cnn_scaled_model.fit(
    x_train_norm, y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
)

cnn_scaled_test_loss, cnn_scaled_test_acc = cnn_scaled_model.evaluate(x_test_norm, y_test)
print(f"Scaled CNN (32-64-128-256) test accuracy: {cnn_scaled_test_acc:.4f}")

## ✅ Task 3 — Train the CNN for 20 Epochs Instead of 10

Same baseline CNN architecture, just given twice the epoch budget, to see
whether validation accuracy keeps rising, levels off, or starts pulling away
from training accuracy — the usual sign that the model is overfitting.


In [ ]:
cnn_20ep_model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation="relu", input_shape=(32, 32, 3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(10, activation="softmax"),
])

cnn_20ep_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

cnn_20ep_history = cnn_20ep_model.fit(
    x_train_norm, y_train,
    epochs=20,
    batch_size=64,
    validation_split=0.1,
)

cnn_20ep_test_loss, cnn_20ep_test_acc = cnn_20ep_model.evaluate(x_test_norm, y_test)
print(f"CNN (20 epochs) test accuracy: {cnn_20ep_test_acc:.4f}")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(cnn_history.history["val_accuracy"], marker="s", label="CNN — 10 epochs")
plt.plot(cnn_20ep_history.history["val_accuracy"], marker="^", label="CNN — 20 epochs")
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("Effect of Training Longer (10 vs 20 Epochs)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## ✅ Task 4 — Add EarlyStopping

Same CNN architecture, given up to 20 epochs, but this time with
`EarlyStopping` watching `val_loss`. Once validation loss stops improving for
`patience` epochs in a row, training halts automatically, and
`restore_best_weights=True` rolls the model back to its strongest checkpoint
rather than whatever epoch happened to be last.


In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1,
)

cnn_es_model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation="relu", input_shape=(32, 32, 3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(10, activation="softmax"),
])

cnn_es_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

cnn_es_history = cnn_es_model.fit(
    x_train_norm, y_train,
    epochs=20,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop],
)

print(f"Training stopped after {len(cnn_es_history.history['loss'])} epochs (cap was 20).")

cnn_es_test_loss, cnn_es_test_acc = cnn_es_model.evaluate(x_test_norm, y_test)
print(f"CNN + EarlyStopping test accuracy: {cnn_es_test_acc:.4f}")

## ✅ Task 5 — Actually Train the Augmented Model

`aug_cnn_model` was only *built* back in the training-strategy section — here
it actually gets trained, for up to 20 epochs with early stopping, so its
results can be compared head-to-head against the non-augmented CNN.


In [ ]:
early_stop_aug = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1,
)

aug_history = aug_cnn_model.fit(
    x_train_norm, y_train,
    epochs=20,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop_aug],
)

aug_test_loss, aug_test_acc = aug_cnn_model.evaluate(x_test_norm, y_test)
print(f"Augmented CNN test accuracy: {aug_test_acc:.4f}")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(cnn_es_history.history["val_accuracy"], marker="s", label="CNN (no augmentation)")
plt.plot(aug_history.history["val_accuracy"], marker="o", label="CNN + Augmentation")
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("Effect of Data Augmentation on Validation Accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# 📊 Full Comparison — Every Model Variant

In [ ]:
final_comparison = pd.DataFrame({
    "Model": [
        "ANN (baseline, 10ep)",
        "CNN (baseline, 10ep)",
        "ANN (deep, Task 1, 10ep)",
        "CNN (scaled filters 32-64-128-256, Task 2, 10ep)",
        "CNN (20ep, Task 3)",
        "CNN + EarlyStopping (Task 4, up to 20ep)",
        "CNN + Augmentation + EarlyStopping (Task 5, up to 20ep)",
    ],
    "Test Accuracy": [
        ann_test_acc,
        cnn_test_acc,
        ann_deep_test_acc,
        cnn_scaled_test_acc,
        cnn_20ep_test_acc,
        cnn_es_test_acc,
        aug_test_acc,
    ],
    "Test Loss": [
        ann_test_loss,
        cnn_test_loss,
        ann_deep_test_loss,
        cnn_scaled_test_loss,
        cnn_20ep_test_loss,
        cnn_es_test_loss,
        aug_test_loss,
    ],
})

final_comparison = final_comparison.sort_values("Test Accuracy", ascending=False).reset_index(drop=True)
final_comparison

In [ ]:
plt.figure(figsize=(11, 5))
plt.barh(final_comparison["Model"], final_comparison["Test Accuracy"], color="steelblue")
plt.xlabel("Test Accuracy")
plt.title("Test Accuracy Across All Model Variants")
plt.gca().invert_yaxis()
for i, v in enumerate(final_comparison["Test Accuracy"]):
    plt.text(v + 0.005, i, f"{v:.3f}", va="center")
plt.tight_layout()
plt.show()

# 🎓 Extension Tasks — Summary

### ✅ What was actually implemented (with real training runs, not just descriptions)
1. **Deepen the ANN** → `ann_deep_model`, 4 hidden layers vs. the baseline's 2
2. **Widen the CNN filters** → baseline already covers 32→64→128; `cnn_scaled_model` pushes it to 32→64→128→256
3. **Train for 20 epochs** → `cnn_20ep_model`, compared directly against the 10-epoch baseline
4. **Add EarlyStopping** → `cnn_es_model`, monitoring `val_loss` with `patience=3` and `restore_best_weights=True`
5. **Train the augmented model for real** → `aug_cnn_model`, trained up to 20 epochs with EarlyStopping

### 🔍 Questions worth answering once you run this
- Does `ann_deep_test_acc` actually beat `ann_test_acc` by much, or is the gain marginal?
- Does `cnn_scaled_test_acc` clearly beat `cnn_test_acc`, or has the extra block hit diminishing returns?
- Looking at the 10- vs 20-epoch curve, does the train/validation gap widen over time?
- How many epochs did `cnn_es_model` actually run before stopping — did EarlyStopping save you compute without costing accuracy?
- Does `aug_test_acc` beat `cnn_es_test_acc`? Augmentation usually narrows the train/val gap and nudges test accuracy up, at the cost of slower epochs.


# ✅ Conclusion

- **The ANN works**, but it's blind to image structure — deepening it (Task 1)
  buys only a modest improvement, since the model never learns that nearby
  pixels are related.
- **The CNN extracts spatial features** through convolution and pooling,
  which is why it clearly beats the ANN at a comparable parameter count.
- **Widening the CNN further** (Task 2) helps, but with diminishing returns —
  a 32×32 image can only be downsampled so many times before there's nothing
  left to pool.
- **Training for longer** (Task 3) raises validation accuracy up to a point,
  after which train and validation accuracy start to diverge — textbook
  overfitting.
- **EarlyStopping** (Task 4) finds a good stopping point automatically,
  saving compute and guarding against that overfitting.
- **Data augmentation** (Task 5) turns out to be one of the strongest, cheapest
  regularizers for a dataset this size, usually improving generalization with
  no architecture changes at all.
- Put together, this pipeline covers the core ideas that show up again and
  again in computer vision interviews and real-world deep learning projects.
